In [0]:
dbutils.secrets.get(scope="ipl-scope", key="storage")

'[REDACTED]'

In [0]:
storage_account_name = "ipl2019"

storage_key = dbutils.secrets.get(
    scope="ipl-scope",
    key="storage"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_key
)

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
target_path = "abfss://silver@ipl2019.dfs.core.windows.net/matches/"

if DeltaTable.isDeltaTable(spark, target_path):
    
    last_load_date = spark.read.format("delta") \
        .load(target_path) \
        .agg({"date": "max"}) \
        .collect()[0][0]

else:
    last_load_date = '1900-01-01'

In [0]:
matches_df = spark.read.csv(
    "abfss://bronze@ipl2019.dfs.core.windows.net/matches/",
    header=True,
    inferSchema=True
).where(col("date") > last_load_date)




In [0]:
matches_clean = matches_df.dropDuplicates()

In [0]:
from delta.tables import DeltaTable
target_path = "abfss://silver@ipl2019.dfs.core.windows.net/matches/"

# Check if table exists
if DeltaTable.isDeltaTable(spark, target_path): # Ensure storage account key is configured properly before this line

    delta_table = DeltaTable.forPath(spark, target_path)

    delta_table.alias("target").merge(
        matches_clean.alias("source"),
        "target.id = source.id"  
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:
    # First time load
    matches_clean.write.format("delta").mode("overwrite").save(target_path)

In [0]:
target_path = "abfss://silver@ipl2019.dfs.core.windows.net/deliveries/"

if DeltaTable.isDeltaTable(spark, target_path):
    
    last_match_id = spark.read.format("delta") \
        .load(target_path) \
        .agg({"match_id": "max"}) \
        .collect()[0][0]

else:
    last_match_id = 0

In [0]:
deliveries_df = spark.read.csv(
    "abfss://bronze@ipl2019.dfs.core.windows.net/deliveries/",
    header=True,
    inferSchema=True
).where(col('match_id') > last_match_id)

In [0]:
deliveries_clean = deliveries_df.dropDuplicates(
    ["match_id", "inning", "over", "ball"]
)

In [0]:
from pyspark.sql.functions import concat_ws

deliveries_clean = deliveries_clean.withColumn(
    "delivery_id",
    concat_ws("_", "match_id", "inning", "over", "ball")
)

In [0]:
from delta.tables import DeltaTable

target_path = "abfss://silver@ipl2019.dfs.core.windows.net/deliveries/"

if DeltaTable.isDeltaTable(spark, target_path):

    delta_table = DeltaTable.forPath(spark, target_path)

    delta_table.alias("target").merge(
        deliveries_clean.alias("source"),
        "target.delivery_id = source.delivery_id"
    ).whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

else:
    deliveries_clean.write.format("delta").mode("overwrite").save(target_path)